# Geison no Google Colab

Este é o fluxo oficial e fino para executar o Geison no Colab. O notebook cuida apenas de ambiente, configuração e chamadas ao CLI; toda a lógica científica continua dentro do pacote `geison-qpcr`.

Execute as células em ordem. Em uma sessão já preparada, a primeira célula atualiza o checkout com `git pull`; em uma sessão nova, ela clona a branch `develop`.


In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d Geison/.git ]; then
  git -C Geison checkout develop
  git -C Geison pull --ff-only origin develop
else
  git clone --branch develop https://github.com/BrunoDCamargo/Geison.git
fi
python -m pip install -e /content/Geison


## Dependências externas

CD-HIT, MAFFT e Primer3 são instalados pelo gerenciador do sistema. O comando `doctor` abaixo confirma o ambiente efetivo antes da análise.


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq cd-hit mafft primer3


In [ ]:
!qpcr-pipeline doctor


## Configuração YAML

O exemplo usa um accession NCBI conhecido e habilita alinhamento e conservação para produzir `report.html`. Troque o target e a seção `input` pelo seu material real antes de uma análise de projeto. O YAML é a fonte de configuração; o notebook não reimplementa nenhum algoritmo científico.


In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/geison_run
cat > /content/geison_run/config.yaml <<'YAML'
target:
  name: SARS-CoV-2-example
input:
  ncbi:
    accessions:
      - NC_045512.2
alignment:
  enabled: true
  threads: 2
conservation:
  enabled: true
  window_size: 100
  step_size: 25
YAML
cat /content/geison_run/config.yaml


## Validar e executar

O `--dry-run` valida configuração, ambiente e plano sem iniciar a análise. A execução real usa um diretório de saída fixo para que os checkpoints possam ser retomados depois.


In [ ]:
!qpcr-pipeline run /content/geison_run/config.yaml --dry-run --outdir /content/geison_run/output


In [ ]:
!qpcr-pipeline run /content/geison_run/config.yaml --outdir /content/geison_run/output


## Retomar uma run

Se a sessão cair ou a execução for interrompida, rode novamente a preparação do ambiente e depois use `--resume` apontando para o mesmo `outdir`. Checkpoints válidos são reutilizados.


In [ ]:
!qpcr-pipeline run /content/geison_run/config.yaml --outdir /content/geison_run/output --resume


## Abrir o relatório

Quando a configuração produzir o relatório consolidado ou de conservação, a célula abaixo abre `/content/geison_run/output/report.html` diretamente no notebook. O arquivo também permanece no diretório de saída para download ou cópia para o Google Drive.

Para atualizar o Geison em uma sessão existente, execute novamente a primeira célula; ela usa `git pull --ff-only origin develop` e reinstala o pacote em modo editável.


In [ ]:
from IPython.display import HTML, display

report_path = "/content/geison_run/output/report.html"
display(HTML(open(report_path, encoding="utf-8").read()))
